# 📖 معالج المفردات — Vocabulary Processor
### لصفحات العمودين (الإنجليزية — العربية)

دفتر متخصص لمعالجة صفحات القاموس ثنائية الأعمدة:

### الميزات:
- استخراج العناوين والتواريخ من الرأس
- فصل العمودين (إنجليزي / عربي)
- التعرف المزدوج (TrOCR + EasyOCR)
- قاعدة بيانات الأنماط (SQLite)
- تحويل التواريخ (ميلادي ↔ هجري)
- ترتيب زمني حسب التواريخ المستخرجة
- تصدير كملف DOCX مرتّب

### الحزم المطلوبة:
- transformers, torch, opencv-python, pillow
- fitz (PyMuPDF), python-dateutil, hijri-converter
- python-docx, gradio

## الخطوة ١: تثبيت الحزم اللازمة

In [ ]:
# ══════════════════════════════════════════════════════════════
# تثبيت جميع الحزم المطلوبة
# ══════════════════════════════════════════════════════════════

# حزم التعلم العميق والتعرف على الحروف
!pip install -q transformers torch opencv-python-headless pillow

# حزم معالجة PDF
!pip install -q fitz  # PyMuPDF

# حزم التعامل مع التواريخ
!pip install -q python-dateutil hijri-converter

# حزم تصدير Word
!pip install -q python-docx

# حزم واجهة المستخدم
!pip install -q gradio

# حزم EasyOCR كاحتياطي
!pip install -q easyocr

print("✅ تمّ تثبيت جميع الحزم بنجاح")

# ── التحقق من التثبيت ──
import torch
import fitz
import gradio as gr
import cv2
import numpy as np
from PIL import Image

from docx import Document
from dateutil import parser as date_parser
from hijri_converter import Gregorian

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ PyMuPDF: {fitz.version}")
print(f"✅ GPU متاح: {torch.cuda.is_available()}")

## الخطوة ٢: قاعدة بيانات الأنماط — PatternDB

In [ ]:
import sqlite3
import re
import json
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, date


# ══════════════════════════════════════════════════════════════
# صنف قاعدة بيانات الأنماط
# ══════════════════════════════════════════════════════════════

@dataclass
class Pattern:
    """نمط نصي مُخزَّن في قاعدة البيانات."""
    id: int
    pattern_type: str  # نوع النمط (تاريخ، عنوان، رقم، إلخ)
    regex: str         # التعبير النمطي
    description: str   # وصف النمط
    examples: str      # أمثلة (JSON)


class PatternDB:
    """قاعدة بيانات SQLite لتخزين أنماط النصوص والتعرف عليها.

    توفّر:
    - تخزين الأنماط النصية (Regex)
    - البحث في النصوص باستخدام الأنماط
    - إدارة الأنماط (إضافة/حذف/تعديل)
    - أنماط جاهزة للتواريخ والعناوين
    """

    def __init__(self, db_path: str = ":memory:"):
        """تهيئة قاعدة البيانات.

        Args:
            db_path: مسار قاعدة البيانات (الذاكرة أو ملف).
        """
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self._create_tables()
        self._seed_default_patterns()

    def _create_tables(self):
        """إنشاء جداول قاعدة البيانات."""
        cursor = self.conn.cursor()

        # جدول الأنماط
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS patterns (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                pattern_type TEXT NOT NULL,
                regex TEXT NOT NULL,
                description TEXT,
                examples TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        # جدول النتائج المطابقة
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS matches (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                pattern_id INTEGER,
                source_text TEXT,
                matched_text TEXT,
                match_start INTEGER,
                match_end INTEGER,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (pattern_id) REFERENCES patterns(id)
            )
        """)

        self.conn.commit()

    def _seed_default_patterns(self):
        """إضافة الأنماط الافتراضية للتواريخ والعناوين."""
        cursor = self.conn.cursor()

        # التحقق من وجود أنماط مسبقًا
        cursor.execute("SELECT COUNT(*) as cnt FROM patterns")
        if cursor.fetchone()['cnt'] > 0:
            return  # البيانات موجودة مسبقًا

        # أنماط التواريخ
        default_patterns = [
            # التاريخ الميلادي بتنسيقات مختلفة
            (
                "تاريخ_ميلادي_شرطة",
                r"\b(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})\b",
                "تاريخ ميلادي بفواصل أو شرطات",
                json.dumps(["15/3/2025", "2025-03-15", "1-12-2024"]),
            ),
            # التاريخ الهجري
            (
                "تاريخ_هجري",
                r"\b(\d{1,2})[-/]?(\d{1,2})[-/]?(\d{4})\s*هـ?\b",
                "تاريخ هجري",
                json.dumps(["15/3/1446", "١٤٤٦/٠٣/١٥", "1446-03-15 هـ"]),
            ),
            # عنوان الصفحة
            (
                "عنوان_صفحة",
                r"^(?:الصفحة|صفحة|Page|ص)\s*[:#]?\s*(\d+)",
                "رقم الصفحة",
            json.dumps(["الصفحة: 15", "ص 42", "Page #7"]),
            ),
            # رقم المجلد أو الجزء
            (
                "رقم_المجلد",
                r"(?:المجلد|الجزء|Volume|Vol)\s*[:#]?\s*(\d+)",
                "رقم المجلد أو الجزء",
                json.dumps(["المجلد: 3", "الجزء 1", "Vol 2"]),
            ),
            # رقم السطر أو البند
            (
                "رقم_البند",
                r"^(?:رقم|البند|Item|No|Item)\s*[.:#]?\s*(\d+)",
                "رقم البند أو العنصر",
                json.dumps(["رقم 1", "البند: 5", "No. 3"]),
            ),
            # حرف الأبجدية الإنجليزية (عنصر قاموس)
            (
                "حرف_إنجليزي",
                r"^\s*([A-Za-z])\s*$",
                "حرف إنجليزي منفرد (عنوان قسم في القاموس)",
                json.dumps(["A", "B", "z"]),
            ),
            # الكلمة الإنجليزية + المعنى (صف القاموس)
            (
                "صف_قاموس",
                r"^([A-Za-z][A-Za-z\s\-']+?)\s+[-–—=]\s*(.+)$",
                "صف قاموس عادي (كلمة إنجليزية - معنى عربي)",
                json.dumps(["apple - تفاحة", "book — كتاب", "cat = قطة"]),
            ),
        ]

        cursor.executemany(
            "INSERT INTO patterns (pattern_type, regex, description, examples) "
            "VALUES (?, ?, ?, ?)",
            default_patterns,
        )
        self.conn.commit()

    def add_pattern(
        self,
        pattern_type: str,
        regex: str,
        description: str = "",
        examples: List[str] = None,
    ) -> int:
        """إضافة نمط جديد إلى قاعدة البيانات.

        Args:
            pattern_type: نوع النمط.
            regex: التعبير النمطي.
            description: وصف النمط.
            examples: أمثلة على النمط.

        Returns:
            معرّف النمط الجديد.
        """
        cursor = self.conn.cursor()
        cursor.execute(
            "INSERT INTO patterns (pattern_type, regex, description, examples) "
            "VALUES (?, ?, ?, ?)",
            (pattern_type, regex, description, json.dumps(examples or [])),
        )
        self.conn.commit()
        return cursor.lastrowid

    def search(
        self,
        text: str,
        pattern_type: Optional[str] = None,
    ) -> List[Dict]:
        """البحث عن الأنماط في النص.

        Args:
            text: النص المراد البحث فيه.
            pattern_type: فلترة بنوع النمط (اختياري).

        Returns:
            قائمة بالنتائج المطابقة.
        """
        cursor = self.conn.cursor()

        if pattern_type:
            cursor.execute(
                "SELECT * FROM patterns WHERE pattern_type = ?",
                (pattern_type,),
            )
        else:
            cursor.execute("SELECT * FROM patterns")

        patterns = cursor.fetchall()
        results = []

        for pattern_row in patterns:
            try:
                matches = re.finditer(pattern_row['regex'], text, re.MULTILINE)
                for match in matches:
                    results.append({
                        "pattern_type": pattern_row['pattern_type'],
                        "matched_text": match.group(0),
                        "start": match.start(),
                        "end": match.end(),
                        "groups": match.groups(),
                    })
            except re.error:
                continue  # تجاوز الأنماط غير الصالحة

        return results

    def get_all_patterns(self) -> List[Pattern]:
        """جلب جميع الأنماط المخزنة."""
        cursor = self.conn.cursor()
        cursor.execute("SELECT * FROM patterns")
        return [
            Pattern(
                id=row['id'],
                pattern_type=row['pattern_type'],
                regex=row['regex'],
                description=row['description'],
                examples=row['examples'],
            )
            for row in cursor.fetchall()
        ]

    def close(self):
        """إغلاق اتصال قاعدة البيانات."""
        self.conn.close()


# ── إنشاء قاعدة بيانات الأنماط ──
pattern_db = PatternDB()
print("✅ تمّ تهيئة قاعدة بيانات الأنماط")
print(f"📊 عدد الأنماط المخزنة: {len(pattern_db.get_all_patterns())}")

## الخطوة ٣: استخراج العناوين والتواريخ

In [ ]:
from datetime import datetime, date
from dateutil import parser as date_parser
from typing import Optional, Tuple


# ══════════════════════════════════════════════════════════════
# دالة استخراج العنوان والتاريخ من رأس الصفحة
# ══════════════════════════════════════════════════════════════

def extract_header_and_date(
    text: str,
    pattern_db_instance: PatternDB,
) -> Dict:
    """استخراج العنوان والتاريخ من نص رأس الصفحة.

    يبحث عن:
        - عنوان الصفحة أو رقمها
        - رقم المجلد أو الجزء
        - التواريخ (ميلادي أو هجري)

    Args:
        text: النص المراد تحليله.
        pattern_db_instance: قاعدة بيانات الأنماط.

    Returns:
        قاموس يحتوي العنوان والتاريخ والمعلومات المستخرجة.
    """
    result = {
        "header_title": "",
        "page_number": None,
        "volume_number": None,
        "date_gregorian": None,
        "date_hijri": None,
        "date_sortable": None,   # للت排序
        "raw_matches": [],
    }

    # ── البحث بالأنماط ──
    matches = pattern_db_instance.search(text)
    result["raw_matches"] = matches

    for match in matches:
        ptype = match["pattern_type"]
        matched = match["matched_text"]

        # استخراج رقم الصفحة
        if ptype == "عنوان_صفحة":
            if match["groups"]:
                try:
                    result["page_number"] = int(match["groups"][0])
                except (ValueError, IndexError):
                    pass

        # استخراج رقم المجلد
        elif ptype == "رقم_المجلد":
            if match["groups"]:
                try:
                    result["volume_number"] = int(match["groups"][0])
                except (ValueError, IndexError):
                    pass

        # استخراج التاريخ الميلادي
        elif ptype == "تاريخ_ميلادي_شرطة":
            parsed_date = _parse_gregorian_date(matched)
            if parsed_date:
                result["date_gregorian"] = parsed_date.strftime("%Y-%m-%d")
                result["date_sortable"] = parsed_date.toordinal()

                # تحويل إلى هجري
                try:
                    hijri = Gregorian.fromdate(parsed_date.date()).to_hijri()
                    result["date_hijri"] = f"{hijri.year}-{hijri.month:02d}-{hijri.day:02d} هـ"
                except Exception:
                    pass

    # ── استخراج العنوان (أول سطر غير فارغ) ──
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if lines:
        # تخطي الأسطر القصيرة جدًا (أرقام صفحات)
        for line in lines:
            if len(line) > 3 and not re.match(r'^\d+$', line):
                result["header_title"] = line
                break

    return result


def _parse_gregorian_date(text: str) -> Optional[datetime]:
    """محاولة تحليل تاريخ ميلادي من النص.

    Args:
        text: نص يحتوي تاريخًا.

    Returns:
        كائن datetime أو None.
    """
    # تنظيف النص
    cleaned = re.sub(r'[هـ]', '', text).strip()

    try:
        parsed = date_parser.parse(cleaned, dayfirst=True, yearfirst=False)
        # التحقق من معقولية التاريخ
        if 1900 <= parsed.year <= 2100:
            return parsed
    except (ValueError, TypeError):
        pass

    # محاولة يدوية
    date_match = re.search(r'(\d{1,2})[-/](\d{1,2})[-/](\d{2,4})', cleaned)
    if date_match:
        day, month, year = date_match.groups()
        try:
            if len(year) == 2:
                year = f"20{year}" if int(year) < 50 else f"19{year}"
            return datetime(int(year), int(month), int(day))
        except ValueError:
            pass

    return None


# ── اختبار الدالة ──
test_text = """القاموس الطبي الحديث
الصفحة: 15 | المجلد: 2
تاريخ: 15/3/2025"""

result = extract_header_and_date(test_text, pattern_db)
print("📋 نتيجة تحليل الرأس:")
print(f"  العنوان:    {result['header_title']}")
print(f"  الصفحة:     {result['page_number']}")
print(f"  المجلد:     {result['volume_number']}")
print(f"  التاريخ:    {result['date_gregorian']}")
print(f"  هجري:      {result['date_hijri']}")

## الخطوة ٤: فصل العمودين

In [ ]:
import cv2
import numpy as np
from typing import List, Tuple, Optional


def split_columns(
    image: np.ndarray,
    num_columns: int = 2,
    min_gap_width: int = 15,
) -> List[np.ndarray]:
    """فصل الصفحة إلى أعمدة متعددة.

    يعتمد على الإسقاط العمودي (Vertical Projection)
    لتحديد الفواصل بين الأعمدة.

    Args:
        image: صورة الصفحة.
        num_columns: العدد المتوقع من الأعمدة.
        min_gap_width: أقل عرض مقبول للفاصل.

    Returns:
        قائمة بصور الأعمدة (من اليسار لليمين).
    """
    # تحويل إلى رمادي
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image.copy()

    # عتبة ثنائية
    _, binary = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # ── الإسقاط العمودي (مجموع البكسلات في كل عمود) ──
    v_proj = np.sum(binary, axis=0)

    # ── تحديد الفواصل بين الأعمدة ──
    # الفاصل هو منطقة متتالية بقيم منخفضة
    width = len(v_proj)
    threshold = np.mean(v_proj) * 0.1  # عتبة الفاصل

    gaps = []
    in_gap = False
    gap_start = 0

    for x, count in enumerate(v_proj):
        if count < threshold and not in_gap:
            in_gap = True
            gap_start = x
        elif count >= threshold and in_gap:
            in_gap = False
            gap_width = x - gap_start
            if gap_width >= min_gap_width:
                # مركز الفاصل
                gap_center = gap_start + gap_width // 2
                gaps.append(gap_center)

    # إذا لم نجد فواصل كافية، نقسم بالتساوي
    if len(gaps) < num_columns - 1:
        # تقسيم متساوٍ مع تجنب الحواف
        margin = int(width * 0.05)
        usable_width = width - 2 * margin
        col_width = usable_width // num_columns
        gaps = []
        for i in range(1, num_columns):
            gaps.append(margin + i * col_width)

    # ── اختيار أفضل فواصل ──
    # إذا وُجدت فواصل أكثر من المطلوب، نختار الأكبر
    if len(gaps) > num_columns - 1:
        # حساب عرض كل فاصل
        gap_widths = []
        for g in gaps:
            left = max(0, g - min_gap_width)
            right = min(width, g + min_gap_width)
            # عرض المنطقة الفارغة حول المركز
            w = right - left
            gap_widths.append((w, g))
        # ترتيب تنازليًا بالعرض واختيار الأكبر
        gap_widths.sort(reverse=True)
        gaps = sorted([g for _, g in gap_widths[:num_columns - 1]])

    # ── قص الأعمدة ──
    boundaries = [0] + gaps + [width]
    columns = []

    for i in range(len(boundaries) - 1):
        left = max(0, boundaries[i])
        right = min(width, boundaries[i + 1])
        col_img = image[:, left:right]
        columns.append(col_img)

    return columns


def identify_columns(
    columns: List[np.ndarray],
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """تحديد العمود الإنجليزي والعربي.

    يعتمد على تحليل اتجاه النص (RTL vs LTR).
    العمود الذي يحتوي نصًا عربيًا (RTL) هو العمود العربي.

    Args:
        columns: قائمة صور الأعمدة.

    Returns:
        (عمود_إنجليزي، عمود_عربي)
    """
    if len(columns) < 2:
        if len(columns) == 1:
            return columns[0], None
        return None, None

    # ── تحديد العمود العربي بالبحث عن أحرف عربية ──
    arabic_char_pattern = re.compile(r'[\u0600-\u06FF]')

    # استخدام EasyOCR للكشف عن اللغات في كل عمود
    # حل بديل: تحليل البكسلات (النص العربي يبدأ من اليمين)
    col_scores = []
    for col_img in columns:
        # تحويل إلى رمادي
        if len(col_img.shape) == 3:
            gray = cv2.cvtColor(col_img, cv2.COLOR_RGB2GRAY)
        else:
            gray = col_img

        # حساب مركز ثقل البكسلات (النص العربي يميل لليمين)
        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )
        v_proj = np.sum(binary, axis=0)

        if np.sum(v_proj) > 0:
            # مركز ثقل أفقي
            x_coords = np.arange(len(v_proj))
            center_of_gravity = np.average(x_coords, weights=v_proj)
            # نسبة المركز (0 = يسار، 1 = يمين)
            ratio = center_of_gravity / len(v_proj)
        else:
            ratio = 0.5

        col_scores.append(ratio)

    # العمود ذو المركز الأيمن (أعلى نسبة) هو العربي (RTL)
    if len(col_scores) >= 2:
        # ترتيب حسب المركز (اليسار أولاً)
        indexed = list(enumerate(col_scores))
        indexed.sort(key=lambda x: x[1])

        # العمود الأول (أيسر) = إنجليزي، الثاني (أيمن) = عربي
        english_idx = indexed[0][0]
        arabic_idx = indexed[-1][0]

        return columns[english_idx], columns[arabic_idx]

    return columns[0], columns[1] if len(columns) > 1 else None


print("✅ تمّ تعريف دوال فصل الأعمدة")

## الخطوة ٥: التعرف المزدوج — MixedOCR (TrOCR + EasyOCR)

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import easyocr
from typing import List, Dict, Optional, Tuple


class MixedOCR:
    """تعرف ضوئي مزدوج باستخدام TrOCR و EasyOCR.

    يجمع بين:
        - TrOCR: دقة عالية للخط اليدوي
        - EasyOCR: دعم أفضل للنص المطبوع واللغات المتعددة

    الاستراتيجية:
        - تشغيل كلا المحركين
        - مقارنة النتائج واختيار الأفضل
        - استخدام قاعدة بيانات الأنماط لتحسين النتائج
    """

    def __init__(
        self,
        trocr_model: str = "microsoft/trocr-base-handwritten",
        easyocr_langs: List[str] = None,
        device: Optional[str] = None,
    ):
        """تهيئة المحرك المزدوج.

        Args:
            trocr_model: اسم نموذج TrOCR.
            easyocr_langs: لغات EasyOCR.
            device: الجهاز (cuda أو cpu).
        """
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        easyocr_langs = easyocr_langs or ['ar', 'en']

        # ── تحميل TrOCR ──
        print(f"⏳ تحميل TrOCR: {trocr_model}")
        self.trocr_processor = TrOCRProcessor.from_pretrained(trocr_model)
        self.trocr_model = VisionEncoderDecoderModel.from_pretrained(trocr_model)
        self.trocr_model.eval()
        if self.device == "cuda":
            self.trocr_model = self.trocr_model.to(self.device)

        # تكوين TrOCR
        self.trocr_model.config.decoder_start_token_id = self.trocr_processor.tokenizer.cls_token_id
        self.trocr_model.config.pad_token_id = self.trocr_processor.tokenizer.pad_token_id

        # ── تحميل EasyOCR ──
        print(f"⏳ تحميل EasyOCR: {easyocr_langs}")
        self.easy_reader = easyocr.Reader(easyocr_langs, gpu=(self.device == "cuda"))

        print(f"✅ تمّ تهيئة المحرك المزدوج على {self.device}")

    def recognize_line(
        self,
        image: np.ndarray,
        language_hint: Optional[str] = None,
    ) -> Dict[str, str]:
        """التعرف على سطر واحد باستخدام المحركين.

        Args:
            image: صورة السطر.
            language_hint: تلميح للغة (ar أو en).

        Returns:
            قاموس يحتوي نتائج كلا المحركين والنتيجة المختارة.
        """
        result = {
            "trocr_text": "",
            "easyocr_text": "",
            "final_text": "",
            "engine": "",
        }

        # ── تحويل الصورة ──
        if len(image.shape) == 2:
            rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        else:
            rgb = image

        # ── TrOCR ──
        try:
            pil_img = image_to_pil(rgb)
            pixel_values = self.trocr_processor(
                pil_img, return_tensors="pt"
            ).pixel_values
            if self.device == "cuda":
                pixel_values = pixel_values.to(self.device)

            with torch.no_grad():
                generated = self.trocr_model.generate(pixel_values)
                trocr_text = self.trocr_processor.tokenizer.batch_decode(
                    generated, skip_special_tokens=True
                )[0]
            result["trocr_text"] = trocr_text.strip()
        except Exception as e:
            result["trocr_text"] = f"[خطأ TrOCR: {e}]"

        # ── EasyOCR ──
        try:
            easy_results = self.easy_reader.readtext(rgb)
            if easy_results:
                easy_text = ' '.join([r[1] for r in easy_results])
            else:
                easy_text = ""
            result["easyocr_text"] = easy_text.strip()
        except Exception as e:
            result["easyocr_text"] = f"[خطأ EasyOCR: {e}]"

        # ── اختيار أفضل نتيجة ──
        trocr_len = len(result["trocr_text"])
        easy_len = len(result["easyocr_text"])

        # اختيار النتيجة الأطول والأوضح
        if trocr_len > 0 and easy_len > 0:
            if trocr_len >= easy_len:
                result["final_text"] = result["trocr_text"]
                result["engine"] = "TrOCR"
            else:
                result["final_text"] = result["easyocr_text"]
                result["engine"] = "EasyOCR"
        elif trocr_len > 0:
            result["final_text"] = result["trocr_text"]
            result["engine"] = "TrOCR"
        elif easy_len > 0:
            result["final_text"] = result["easyocr_text"]
            result["engine"] = "EasyOCR"

        return result

    def recognize_column(
        self,
        column_image: np.ndarray,
    ) -> List[Dict]:
        """التعرف على جميع أسطر عمود واحد.

        Args:
            column_image: صورة العمود.

        Returns:
            قائمة بنتائج الأسطر.
        """
        # تقسيم إلى أسطر
        lines = segment_lines_simple(column_image)

        results = []
        for line_img in lines:
            line_result = self.recognize_line(line_img)
            line_result["line_image"] = line_img
            results.append(line_result)

        return results


def image_to_pil(rgb_array: np.ndarray):
    """تحويل مصفوفة RGB إلى صورة PIL."""
    return Image.fromarray(rgb_array)


def segment_lines_simple(image: np.ndarray) -> List[np.ndarray]:
    """تقسيم بسيط للصورة إلى أسطر.

    Args:
        image: صورة الإدخال.

    Returns:
        قائمة بصور الأسطر.
    """
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image.copy()

    _, binary = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    h_proj = np.sum(binary, axis=1)

    lines = []
    in_line = False
    start_y = 0

    for y, count in enumerate(h_proj):
        if count > 0 and not in_line:
            in_line = True
            start_y = y
        elif count == 0 and in_line:
            in_line = False
            if y - start_y >= 5:
                lines.append((start_y, y))

    if in_line and len(h_proj) - start_y >= 5:
        lines.append((start_y, len(h_proj)))

    line_images = []
    for y1, y2 in lines:
        line_images.append(image[max(0, y1-2):min(image.shape[0], y2+2), :])

    return line_images


print("⏳ جاري تهيئة المحرك المزدوج...")
mixed_ocr = MixedOCR()
print("✅ المحرك المزدوج جاهز")

## الخطوة ٦: المعالجة الكاملة — تحميل الملف واستخراج المفردات

In [ ]:
import fitz  # PyMuPDF
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from typing import List, Dict


# ══════════════════════════════════════════════════════════════
# معالجة ملف كامل (PDF أو صور)
# ══════════════════════════════════════════════════════════════

def process_file(
    file_path: str,
    ocr_engine: MixedOCR,
    pattern_db_inst: PatternDB,
) -> List[Dict]:
    """معالجة ملف PDF أو صور واستخراج المفردات.

    الخطوات:
        1. تحويل الملف إلى صور
        2. استخراج الرأس والتاريخ من كل صفحة
        3. فصل العمودين
        4. التعرف على النصوص
        5. تجميع النتائج

    Args:
        file_path: مسار الملف.
        ocr_engine: محرك OCR المزدوج.
        pattern_db_inst: قاعدة بيانات الأنماط.

    Returns:
        قائمة بقواميس كل صفحة.
    """
    file_path = Path(file_path)
    all_pages = []

    # ── تحويل الملف إلى صور ──
    if file_path.suffix.lower() == '.pdf':
        doc = fitz.open(str(file_path))
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            mat = fitz.Matrix(300 / 72, 300 / 72)
            pix = page.get_pixmap(matrix=mat)
            img_data = pix.samples
            img_array = np.frombuffer(img_data, dtype=np.uint8)
            img_array = img_array.reshape((pix.height, pix.width, pix.n))
            if pix.n == 4:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_RGBA2RGB)
            elif pix.n == 1:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
            all_pages.append((page_num + 1, img_array))
        doc.close()
    else:
        img = cv2.imread(str(file_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        all_pages.append((1, img))

    # ── معالجة كل صفحة ──
    results = []
    for page_num, page_img in all_pages:
        print(f"📄 صفحة {page_num}...")

        # استخراج السطور الأولى كرأس
        header_lines = segment_lines_simple(page_img)
        header_text = ""
        if header_lines:
            # التعرف على أول سطرين كرأس
            for hl in header_lines[:2]:
                h_result = ocr_engine.recognize_line(hl)
                header_text += h_result["final_text"] + "\n"

        # استخراج معلومات الرأس والتاريخ
        header_info = extract_header_and_date(header_text, pattern_db_inst)

        # فصل الأعمدة
        columns = split_columns(page_img, num_columns=2)
        english_col, arabic_col = identify_columns(columns)

        # التعرف على النصوص
        english_entries = []
        arabic_entries = []

        if english_col is not None:
            en_results = ocr_engine.recognize_column(english_col)
            english_entries = [r["final_text"] for r in en_results if r["final_text"]]

        if arabic_col is not None:
            ar_results = ocr_engine.recognize_column(arabic_col)
            arabic_entries = [r["final_text"] for r in ar_results if r["final_text"]]

        page_data = {
            "page_number": page_num,
            "header": header_info,
            "english_entries": english_entries,
            "arabic_entries": arabic_entries,
            "date_sortable": header_info.get("date_sortable"),
        }
        results.append(page_data)

    # ── ترتيب زمنيًا حسب التواريخ ──
    results.sort(key=lambda x: x["date_sortable"] or 0)

    print(f"\n✅ تمّ معالجة {len(results)} صفحة")
    return results


# ══════════════════════════════════════════════════════════════
# تصدير إلى DOCX مرتّب
# ══════════════════════════════════════════════════════════════

def export_to_docx(
    pages_data: List[Dict],
    output_path: str = "/content/vocab_output.docx",
):
    """تصدير النتائج إلى ملف Word مرتّب.

    Args:
        pages_data: قائمة بيانات الصفحات.
        output_path: مسار ملف DOCX الناتج.
    """
    doc = Document()

    # عنوان المستند
    title = doc.add_heading("معجم المفردات — مرتّب زمنيًا", level=1)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # تاريخ الإنشاء
    doc.add_paragraph(
        f"تاريخ الإنشاء: {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    ).alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph()  # سطر فارغ

    for page in pages_data:
        header = page["header"]

        # عنوان الصفحة
        heading_text = f"صفحة {page['page_number']}"
        if header.get("header_title"):
            heading_text += f" — {header['header_title']}"
        if header.get("date_gregorian"):
            heading_text += f" ({header['date_gregorian']})"

        doc.add_heading(heading_text, level=2)

        # جدول المفردات
        if page["english_entries"] or page["arabic_entries"]:
            table = doc.add_table(rows=1, cols=2)
            table.style = 'Table Grid'

            # عنوان الجدول
            hdr_cells = table.rows[0].cells
            hdr_cells[0].text = "English"
            hdr_cells[1].text = "العربية"

            # جعل العنوان بالخط العريض
            for cell in hdr_cells:
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.bold = True

            # إضافة المفردات
            max_rows = max(
                len(page["english_entries"]),
                len(page["arabic_entries"]),
            )

            for row_idx in range(max_rows):
                row_cells = table.add_row().cells

                # الإنجليزية
                if row_idx < len(page["english_entries"]):
                    row_cells[0].text = page["english_entries"][row_idx]
                else:
                    row_cells[0].text = ""

                # العربية
                if row_idx < len(page["arabic_entries"]):
                    row_cells[1].text = page["arabic_entries"][row_idx]
                else:
                    row_cells[1].text = ""

        doc.add_paragraph()  # سطر فارغ بين الصفحات

    # حفظ الملف
    doc.save(output_path)
    print(f"✅ تمّ تصدير المستند: {output_path}")

    return output_path


print("✅ تمّ تعريف دوال المعالجة والتصدير")

## الخطوة ٧: واجهة Gradio التفاعلية

In [ ]:
import gradio as gr
import tempfile


# ══════════════════════════════════════════════════════════════
# دوال واجهة المستخدم
# ══════════════════════════════════════════════════════════════

def process_and_export(file_obj):
    """معالجة الملف وتصدير DOCX.

    Args:
        file_obj: كائن الملف المُحمَّل.

    Returns:
        (ملخص النتائج، مسار ملف DOCX)
    """
    if file_obj is None:
        return "❌ لم يتم رفع ملف. يرجى اختيار ملف PDF أو صورة.", None

    file_path = file_obj.name if hasattr(file_obj, 'name') else file_obj

    try:
        # معالجة الملف
        pages_data = process_file(
            file_path=file_path,
            ocr_engine=mixed_ocr,
            pattern_db_inst=pattern_db,
        )

        # تصدير إلى DOCX
        docx_path = export_to_docx(pages_data, "/content/vocab_output.docx")

        # إنشاء ملخص
        summary_lines = [
            f"✅ تمّت المعالجة بنجاح!",
            f"",
            f"📊 الإحصائيات:",
            f"  عدد الصفحات: {len(pages_data)}",
        ]

        for page in pages_data:
            header = page["header"]
            en_count = len(page["english_entries"])
            ar_count = len(page["arabic_entries"])
            date_str = header.get("date_gregorian", "غير محدد")
            summary_lines.append(
                f"  صفحة {page['page_number']}: {en_count} إنجليزية, {ar_count} عربية (تاريخ: {date_str})"
            )

        summary = "\n".join(summary_lines)
        return summary, docx_path

    except Exception as e:
        return f"❌ خطأ في المعالجة: {e}", None


# ══════════════════════════════════════════════════════════════
# بناء الواجهة
# ══════════════════════════════════════════════════════════════

with gr.Blocks(title="معالج المفردات", theme=gr.themes.Soft()) as vocab_ui:
    gr.Markdown("# 📖 معالج المفردات — Vocabulary Processor")
    gr.Markdown("### لصفحات العمودين (الإنجليزية — العربية)")
    gr.Markdown(
        "حمّل ملف PDF أو صورة تحتوي صفحات قاموس ثنائية الأعمدة. "
        "سيتم استخراج المفردات والترتيب زمنيًا وتصديرها كملف Word."
    )

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label="📁 رفع الملف",
                file_types=[".pdf", ".png", ".jpg", ".jpeg", ".tiff"],
            )
            process_btn = gr.Button(
                "🚀 معالجة وتصدير",
                variant="primary",
                size="lg",
            )

    summary_output = gr.Textbox(
        label="📋 ملخص النتائج",
        interactive=False,
        lines=10,
    )

    docx_output = gr.File(
        label="📥 ملف Word المُصدَّر (DOCX)",
        file_types=[".docx"],
    )

    # ربط الحدث
    process_btn.click(
        fn=process_and_export,
        inputs=[file_input],
        outputs=[summary_output, docx_output],
    )


print("✅ تمّ بناء واجهة Gradio بنجاح")

## الخطوة ٨: تشغيل الواجهة

In [ ]:
# تشغيل واجهة Gradio مع رابط مشاركة عام
vocab_ui.launch(share=True, debug=True)